In [4]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

smoothed_data = pd.read_csv('smoothed_close_prices.csv')

# Δημιουργία καθυστερημένων τιμών (lags) για τη στήλη 'Smoothed Close Price'
for lag in range(1, 5):  # Δημιουργία 3 καθυστερημένων τιμών
    smoothed_data[f'smoothed_close_t-{lag}'] = smoothed_data['Smoothed Close Price'].shift(-lag)

smoothed_data.fillna(0, inplace=True)

smoothed_data.head()

,Date,Close Price,Smoothed Close Price,smoothed_close_t-1,smoothed_close_t-2,smoothed_close_t-3,smoothed_close_t-4
0,2024-11-13,225.12,225.405344,225.466487,225.589532,225.775824,226.026544
1,2024-11-12,224.23,225.466487,225.589532,225.775824,226.026544,226.341164
2,2024-11-11,224.23,225.589532,225.775824,226.026544,226.341164,226.716854
3,2024-11-08,226.96,225.775824,226.026544,226.341164,226.716854,227.147451
4,2024-11-07,227.48,226.026544,226.341164,226.716854,227.147451,227.623096


In [5]:
smoothed_data['Date'] = pd.to_datetime(smoothed_data['Date'])

# Χωρίζουμε τα δεδομένα σε εκπαίδευση και επικύρωση
validation_data = smoothed_data[smoothed_data['Date'].dt.year == 2024]  # Στοιχεία για επικύρωση (2024)
training_data = smoothed_data[smoothed_data['Date'].dt.year < 2024]     # Στοιχεία για εκπαίδευση (προηγούμενα έτη)

# Ορισμός χαρακτηριστικών και στόχου
X_train = training_data[['smoothed_close_t-1', 'smoothed_close_t-2', 'smoothed_close_t-3']]
y_train = training_data['Smoothed Close Price']

X_val = validation_data[['smoothed_close_t-1', 'smoothed_close_t-2', 'smoothed_close_t-3']]
y_val = validation_data['Smoothed Close Price']

In [6]:
# Πολυωνυμικά χαρακτηριστικά 
poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_val_poly = poly.transform(X_val)

# Υπερπαράμετροι για L1 (Lasso) και L2 (Ridge) κανονικοποίηση
alpha_l1 = 0.1  # Για το Lasso
alpha_l2 = 0.1 # Για το Ridge

# Μοντέλο Lasso (L1 κανονικοποίηση)
lasso = Lasso(alpha=alpha_l1, max_iter=10000)
lasso.fit(X_train_poly, y_train)
predictions_train_lasso = lasso.predict(X_train_poly)
predictions_val_lasso = lasso.predict(X_val_poly)

# Μοντέλο Ridge (L2 κανονικοποίηση)
ridge = Ridge(alpha=alpha_l2)
ridge.fit(X_train_poly, y_train)
predictions_train_ridge = ridge.predict(X_train_poly)
predictions_val_ridge = ridge.predict(X_val_poly)

# Υπολογισμός μετρικών σφάλματος για το Lasso και το Ridge
# Lasso
mse_train_lasso = mean_squared_error(y_train, predictions_train_lasso)
mae_train_lasso = mean_absolute_error(y_train, predictions_train_lasso)

mse_val_lasso = mean_squared_error(y_val, predictions_val_lasso)
mae_val_lasso = mean_absolute_error(y_val, predictions_val_lasso)

# Ridge
mse_train_ridge = mean_squared_error(y_train, predictions_train_ridge)
mae_train_ridge = mean_absolute_error(y_train, predictions_train_ridge)

mse_val_ridge = mean_squared_error(y_val, predictions_val_ridge)
mae_val_ridge = mean_absolute_error(y_val, predictions_val_ridge)

# Εκτύπωση των μετρικών σφάλματος και παραμέτρων
print("Lasso (L1) Regression Metrics:")
print(f"Training MSE: {mse_train_lasso}, MAE: {mae_train_lasso}")
print(f"Validation MSE: {mse_val_lasso}, MAE: {mae_val_lasso}")

print("\nRidge (L2) Regression Metrics:")
print(f"Training MSE: {mse_train_ridge}, MAE: {mae_train_ridge}")
print(f"Validation MSE: {mse_val_ridge}, MAE: {mae_val_ridge}")

Lasso (L1) Regression Metrics:
Training MSE: 1.9937400319932388, MAE: 0.3724244722822611
Validation MSE: 0.11039714810327643, MAE: 0.27415839489885446

Ridge (L2) Regression Metrics:
Training MSE: 1.2192229949730726, MAE: 0.05943228172250188
Validation MSE: 0.00126063623223442, MAE: 0.028550916793013477
